In [1]:
import os
import copy
import numpy as np
import pandas as pd
import geopandas as gpd

from pathlib import Path

from src import utils, stats_utils
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_context("paper", font_scale=1.25)

from src.data.lfs import EuLfs
from src.data.framework import Esco, Classifications

# load paths
useful_paths = utils.UsefulPaths()

%load_ext autoreload
%autoreload 2

#### Read data

##### Covariates
- NACE labels
- GBN shares based on unweighted classifications
- Earnings deciles data

NACE labels

In [2]:
# for NACE labels
classifications = Classifications()
covariates_by_nace = classifications.nace_1d

ISO codes

In [3]:
iso_codes = pd.read_excel(
    os.path.join(useful_paths.data_raw, "metadata", "Country_Codes_and_Names.xlsx"),
)

col_repl = {
    "COUNTRY NAME": {
        "Germany (including former GDR from 1991)": "Germany",
    }
}

iso_codes.replace(col_repl, inplace=True)
iso_codes

,AREA,CODE,COUNTRY NAME
0,European Union (EU),EU-28,European Union (28 countries)
1,European Union (EU),BE,Belgium
2,European Union (EU),BG,Bulgaria
3,European Union (EU),CZ,Czech Republic
4,European Union (EU),DK,Denmark
5,European Union (EU),DE,Germany
6,European Union (EU),EE,Estonia
7,European Union (EU),IE,Ireland
8,European Union (EU),EL,Greece
9,European Union (EU),ES,Spain


##### Country-specific earnings deciles (DE)

In [4]:
# earnings deciles
earnings_deciles = pd.read_csv(
    os.path.join(useful_paths.data_raw, "metadata", "earnings_per_deciles.csv"),
    # dtype={"decile": str}
)

# percent increase between deciles
perc_increase_by_decile = (
    earnings_deciles.annual_earnings / earnings_deciles.annual_earnings.shift(1)
)
pcinc_9_to_10 = perc_increase_by_decile.tail(1).values[0]
perc_increase_by_decile
earnings_deciles

,country,decile,annual_earnings,year
0,DE,1,7635,2016
1,DE,2,11970,2016
2,DE,3,14817,2016
3,DE,4,17256,2016
4,DE,5,19692,2016
5,DE,6,22243,2016
6,DE,7,25088,2016
7,DE,8,28832,2016
8,DE,9,34299,2016
9,DE,10,55368,2016


##### Read Eurostat SILC earnings data
https://www.google.com/url?sa=t&rct=j&q=&esrc=s&source=web&cd=&ved=2ahUKEwiI__7O_sP6AhXbgv0HHXBXDagQFnoECBQQAQ&url=https%3A%2F%2Fec.europa.eu%2Feurostat%2Fstatistics-explained%2Fimages%2F9%2F9f%2FCountry_Codes_and_Names.xlsx&usg=AOvVaw3E1WY8Fd33SHLq3c0FM01U

https://ec.europa.eu › Country_Codes_and_Names

In [5]:
keys = [
    "Sheet 1",
    "Sheet 2",
    "Sheet 3",
    "Sheet 4",
    "Sheet 5",
    "Sheet 6",
    "Sheet 7",
    "Sheet 8",
    "Sheet 9",
    "Sheet 10",
]

eusilc = pd.read_excel(
    os.path.join(
        useful_paths.data_raw, "metadata", "ilc_di01__custom_3490460_spreadsheet.xlsx"
    ),
    sheet_name=keys,
    header=10,
    nrows=46,
    na_values=":",
)

col_repl = {
    "country": {
        "Germany (until 1990 former territory of the FRG)": "Germany",
        "Kosovo (under United Nations Security Council Resolution 1244/99)": "Kosovo",
        "Czechia": "Czech Republic",
    }
}

eusilc_fmt = {}
for k, df in eusilc.items():
    k_new = k.strip("Sheet ")
    df.columns = ["country", "annual_earnings", "flag"]
    df["decile"] = int(k_new)
    df.replace(to_replace=col_repl, inplace=True)
    eusilc_fmt[k_new] = df

C:\Users\fzaussinger\Miniconda3\envs\re4gt\lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [6]:
# concat and sort
eusilc_final = pd.concat(list(eusilc_fmt.values()))
eusilc_final = eusilc_final.sort_values(["country", "decile"])


# impute tenth decile based on jump from 9th to 10th in German data (VERY CRUDE)
dfs = []
for i, df in eusilc_final.groupby("country"):
    df.iloc[-1, 1] = df.iloc[-2, 1] * pcinc_9_to_10
    dfs.append(df)

eusilc_imp = pd.concat(dfs)
eusilc_imp_final = eusilc_imp.merge(
    iso_codes, left_on="country", right_on="COUNTRY NAME", how="left"
).drop(["AREA", "COUNTRY NAME", "flag"], axis=1)
eusilc_imp_final = eusilc_imp_final.rename(columns={"CODE": "country_code"})
eusilc_imp_final["decile"] = eusilc_imp_final["decile"].astype(float)

# replace whitespace
eusilc_imp_final["country_code"] = eusilc_imp_final["country_code"].str.strip()

In [7]:
eusilc_imp_final["country_code"].unique()

array([nan, 'AT', 'BE', 'BG', 'HR', 'CY', 'CZ', 'DK', 'EE', 'FI', 'FR',
       'DE', 'EL', 'HU', 'IS', 'IE', 'IT', 'LV', 'LT', 'LU', 'MT', 'NL',
       'NO', 'PL', 'PT', 'RO', 'SK', 'SI', 'ES', 'SE', 'CH', 'TR', 'UK'],
      dtype=object)

GBN shares and categories based on unweighted classifications

In [11]:
# GBN shares
esco = Esco()
ndigits = 3

gbn_shares_no_wt = esco.read_gbn_classification(agg_to_isco_at_digit=ndigits)

gbn_shares_no_wt = gbn_shares_no_wt.rename(
    columns={"preferredLabel_isco": "ISCO3D_label"}
)
gbn_shares_no_wt["NOBS"] = 1

# define the GBN category of an ISCO 3D group as the one with the highest fraction
gbn_shares_no_wt["category_sl"] = gbn_shares_no_wt[
    ["share_green", "share_brown_sl", "share_neutral_sl"]
].idxmax(axis=1)

gbn_shares_no_wt["category_slt"] = gbn_shares_no_wt[
    ["share_green", "share_brown_slt", "share_neutral_slt"]
].idxmax(axis=1)

gbn_shares_no_wt = gbn_shares_no_wt.replace(
    to_replace={
        "category_sl": {
            "share_green": "green",
            "share_brown_sl": "brown",
            "share_neutral_sl": "neutral",
        },
        "category_slt": {
            "share_green": "green",
            "share_brown_slt": "brown",
            "share_neutral_slt": "neutral",
        },
    }
)

gbn_shares_no_wt.to_csv(
    os.path.join(
        useful_paths.data_processed,
        "esco",
        "final_gbn_shares_by_isco{}d_unweighted.csv".format(ndigits),
    )
)
assert 1 == 2

AssertionError: 

##### LFS data

In [ ]:
reprocess = False

config_file = "eu_lfs_config.yml"
config = utils.load_config(os.path.join(useful_paths.config_dir, config_file))
eulfs = EuLfs(config=config)

In [ ]:
year = 2019

countries_isco3d = [
    "AT",
    "BE",
    "CH",
    "CY",
    "CZ",
    "DE",
    "DK",
    "EE",
    "ES",
    "FI",
    "FR",
    "GR",
    "HR",
    "HU",
    "IE",
    "IS",
    "IT",
    "LT",
    "LU",
    "LV",
    "NL",
    "NO",
    "PT",
    "RO",
    "SE",
    "SK",
    "UK",
]

if reprocess:
    eulfs.preprocess_files(
        years=[year],
        countries=countries_isco3d,
        save_file=True,
        optional_output_dir=config["paths"]["interim"],
        output_fname="eu_lfs_merged_{year}",
    )

In [ ]:
df_all = eulfs.read_preprocessed_file(year=year)
df_all

#### Descriptive statistics

**Sample size by sector and country**
Sectoral focus on key sectors affected by green transition

In [ ]:
sample_size_by_ind_and_country = (
    df_all.groupby(["NACE1D", "COUNTRY"]).count()["COEFF"].unstack()
)
sample_size_by_ind_and_country.to_csv(
    os.path.join(
        useful_paths.figure_dir, "03_eulfs", "eu_lfs_sample_size_by_ind_and_country.csv"
    )
)
sample_size_by_ind_and_country

**Number of unique occupation groups by sector and country**
Sectoral focus on key sectors affected by green transition

In [ ]:
n_unique_occ_by_ind_and_country = (
    df_all.groupby(["NACE1D", "COUNTRY"])["ISCO3D"].nunique().unstack()
)
n_unique_occ_by_ind_and_country.to_csv(
    os.path.join(
        useful_paths.figure_dir, "03_eulfs", "n_unique_occ_by_ind_and_country.csv"
    )
)
n_unique_occ_by_ind_and_country

##### Regional distribution of unique occupations
Needed for regional reskilling constraint.

In Germany, over 90% of all unique 3-digit occupations have COEFF > 0 in each NUTS2 region

In [ ]:
df = df_all[df_all.COUNTRYW == "DE"]
n_isco_max = df.ISCO.unique().shape[0]
occ_number_by_nuts2_and_isco3 = df.groupby(["NUTS_ID", "ISCO3D"]).aggregate(
    {"COEFF": np.sum}
)
occ_number_by_nuts2_and_isco3

n_unique_occ_by_nuts2 = (
    occ_number_by_nuts2_and_isco3.reset_index()
    .groupby("NUTS_ID")
    .aggregate({"COEFF": np.count_nonzero})
)
n_unique_occ_by_nuts2 = n_unique_occ_by_nuts2.rename(columns={"COEFF": "n_unique_abs"})
n_unique_occ_by_nuts2["n_unique_rel"] = (
    n_unique_occ_by_nuts2["n_unique_abs"] / n_isco_max
)
n_unique_occ_by_nuts2

##### Missing earnings information (INCDECIL) per country
Many countries totally lack any data on earnings

In [ ]:
missing_lfs_data_incdecile = (
    df_all.groupby("COUNTRYW")["INCDECIL"]
    .apply(utils.perc_missing)
    .reset_index()
    .drop(labels="level_1", axis=1)
)
missing_lfs_data_incdecile = missing_lfs_data_incdecile.sort_values(
    "missing_obs_rel", ascending=False
)
missing_lfs_data_incdecile.to_csv(
    os.path.join(
        useful_paths.figure_dir,
        "03_eulfs",
        "eulfs_perc_missing_incdecile_per_country.csv",
    )
)
missing_lfs_data_incdecile

##### Imputation of INCDECIL variable
We therefore impute missing deciles by country-specific occupuation-industry medians

In [ ]:
# calculate median income decile per country-occupation-industry group
df_all["INCDECIL"] = df_all["INCDECIL"].astype(float)
incdecil_by_cnt_isco_nace = (
    df_all.groupby(["COUNTRYW", "ISCO3D", "NACE1D"])
    .aggregate({"INCDECIL": np.nanmedian})
    .reset_index()
)
incdecil_by_cnt_isco_nace = incdecil_by_cnt_isco_nace.rename(
    columns={"INCDECIL": "INCDECIL_median"}
)
incdecil_by_cnt_isco_nace["INCDECIL_median_floor"] = np.floor(
    incdecil_by_cnt_isco_nace["INCDECIL_median"]
)
incdecil_by_cnt_isco_nace["INCDECIL_median_ceil"] = np.ceil(
    incdecil_by_cnt_isco_nace["INCDECIL_median"]
)

# join to LFS data
df_all = pd.merge(
    df_all, incdecil_by_cnt_isco_nace, on=["COUNTRYW", "ISCO3D", "NACE1D"]
)

# impute with ceiled medians (more conservative estimate)
df_all["INCDECIL_imputed"] = df_all["INCDECIL"].fillna(df_all["INCDECIL_median_ceil"])

Save overview of imputation impact

In [ ]:
incdecil_imp_results = df_all.groupby("COUNTRYW")[
    ["INCDECIL", "INCDECIL_imputed"]
].apply(utils.perc_missing)
incdecil_imp_results.to_csv(
    os.path.join(
        useful_paths.figure_dir,
        "03_eulfs",
        "eulfs_incdecile_per_country_after_imputing.csv",
    )
)
incdecil_imp_results

Save file

In [ ]:
utils.save_df_to_files(
    df_all,
    output_dir=eulfs.path_eulfs_interim,
    fname_no_ext="eu_lfs_merged_{year}_incdecil_imputed".format(year=year),
)

In [ ]:
list(eusilc_imp_final.country_code.unique())

In [ ]:
eusilc_imp_final

In [ ]:
list(df_all.COUNTRYW.unique())

#### Data preprocessing

Countries with data limitations, occupations:
•	ISCO-08 2D: BG, PL
•	ISCO-08 1D: MT

Countries with data limitations, regions:
•	NUTS 1: UK
•	NUTS Suppressed: NL

Countries with data limitations, income:
- AT, ES, SE, NO, CZ, IS

Countries removed for ISCO-08 3D results:
- BG
- PL
- MT

Reasons for dropping observations:
- ISCO not coded at 3D (but at 2 or 1D)
- COEFF is NAN
- ISCO3D is 633 (subsistence farming --> investigate: https://esco.ec.europa.eu/en/classification/occupation?uri=http://data.europa.eu/esco/isco/C633)

##### Attach covariates
- NACE labels
- GBN shares
- earnings deciles

Attach NACE labels and GBN shars

In [ ]:
eulfs.join_covariates(
    year=2019,
    optional_input_dir=config["paths"]["interim"],
    input_fname_lfs="eu_lfs_merged_{year}_incdecil_imputed",
    covariates_by_isco=gbn_shares_no_wt,
    covariates_by_nace=covariates_by_nace,
    optional_output_dir=config["paths"]["interim"],
    output_fname_lfs="eu_lfs_merged_{year}_with_final_unweighted_shares_incdecil_imputed",
    isco_covariate_selection=None,
)

Match country-specific earnings distributions based on deciles

In [ ]:
eulfs.join_covariates(
    year=2019,
    optional_input_dir=config["paths"]["interim"],
    input_fname_lfs="eu_lfs_merged_{year}_with_final_unweighted_shares_incdecil_imputed",
    covariates_by_isco=eusilc_imp_final,
    isco_join_col_eulfs=["COUNTRYW", "INCDECIL_imputed"],
    isco_join_col_covariates=["country_code", "decile"],
    optional_output_dir=config["paths"]["interim"],
    output_fname_lfs="eu_lfs_merged_{year}_with_final_unweighted_shares_and_earnings_incdecil_imputed",
    isco_covariate_selection=None,
)

##### Aggregate by regions & industries

In [ ]:
# aggregate
agg_dict = {
    "COEFF": np.sum,
    "NOBS": np.sum,
    "COEFF_share_green": np.sum,
    "COEFF_share_brown_sl": np.sum,
    "COEFF_share_brown_slt": np.sum,
}

if reprocess:
    # by 1-digit industry
    eulfs.aggregate(
        year=2019,
        group_by=["NACE1D_label"],
        agg_dict=agg_dict,
        input_fname="eu_lfs_merged_{year}_with_final_unweighted_shares",
        output_fname="eulfs_{year}_by_{by}_final_unweighted_shares",
    )

    # by 1-digit industry and country
    eulfs.aggregate(
        year=2019,
        group_by=["NACE1D_label", "COUNTRYW"],
        agg_dict=agg_dict,
        input_fname="eu_lfs_merged_{year}_with_final_unweighted_shares",
        output_fname="eulfs_{year}_by_{by}_final_unweighted_shares",
    )

    # by nuts-2 regions and 1-digit industries
    eulfs.aggregate(
        year=2019,
        group_by=["NUTS_ID", "NACE1D_label"],
        agg_dict=agg_dict,
        input_fname="eu_lfs_merged_{year}_with_final_unweighted_shares",
        output_fname="eulfs_{year}_by_{by}_final_unweighted_shares",
    )